## PyNS Turorial: Getting started...
Authored by: Abdallah Alashqar (abdallah.j.alashqar@fau.de)

First authored on: 04.06.2025

In [ ]:
import numpy as np
import h5py
# add parent directory to path
import sys
import os

# Add parent directory to path to import pyns modules
current_dir = os.path.dirname(os.path.abspath('__file__'))
pyns_root = os.path.join(current_dir, '..')
sys.path.insert(0, pyns_root)

from pyns.utils import filter_axon_trajectories, create_single_pulse_waveform
from pyns.axon_models import MyelinatedAxon

import matplotlib.pyplot as plt
import pyvista as pv

### Step 1: load the main resources for your simulation (field dict and axon fibers)
- A `field_dict` is a dictionary containing 4 entries:
    - `x`, `y` and `z`: numpy arrays represnting the x, y and z axis points respectively of a rectilinear grid (in meters).
    - `field_values`: a numpy aray of shape (len(x), len(y), len(z)) representing the potential value at each point of the rectilinear grid.
- After loading the field dict we convert the coordinates from m to um since these will be used for interpolating voltage values around axons which have coordinates in um
- We then define ranges with a safety margin of 1mm to filter axon trajectories and make sure all interpolation points are within the field


In [ ]:
# Load the field dictionary from the test dataset
# Get pyns root directory
pyns_root_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Path to test dataset files
field_path = os.path.join(pyns_root_dir, 'src', 'pyns', 'test_dataset', 'lumbar-tSCS_cathode_T11-T12_anode_navel-sides_units_V_m_cropped.h5')
axons_path = os.path.join(pyns_root_dir, 'src', 'pyns', 'test_dataset', 'RightSoleusAxons_diams_from_Schalow1992_cropped.npy')

# Load field dictionary from HDF5 file
with h5py.File(field_path, 'r') as f:
    field_dict = {key: f[key][()] for key in f.keys()}

# Convert coordinates from m to um for consistency with axon coordinates
field_dict["x"] *= 1e6  # m to um
field_dict["y"] *= 1e6  # m to um
field_dict["z"] *= 1e6  # m to um

print(f"Field dictionary entries:")
for key in field_dict.keys():
    print(f"  {key}: shape={field_dict[key].shape}, dtype={field_dict[key].dtype}")

# Define spatial ranges for filtering axon trajectories (with 1mm safety margin)
x_range = [field_dict["x"].min() + 1000, field_dict["x"].max() - 1000]
y_range = [field_dict["y"].min() + 1000, field_dict["y"].max() - 1000]
z_range = [field_dict["z"].min() + 1000, field_dict["z"].max() - 1000]

print(f"\nSpatial ranges for filtering (um):")
print(f"  x: {x_range}")
print(f"  y: {y_range}")
print(f"  z: {z_range}")

# Load and filter axon trajectories
fibers_dict = np.load(axons_path, allow_pickle=True)[()]
print(f"\nNumber of fibers before filtering: {len(fibers_dict)}")

# Minimum axon length: fibers shorter than this are unlikely to show meaningful responses in tSCS
min_axon_length = 40.0 * 1e3  # in micrometers
axon_dicts = filter_axon_trajectories(
    fibers_dict,
    x_range=x_range,
    y_range=y_range,
    z_range=z_range,
    min_axon_length=min_axon_length,
    rank=0,
)
print(f"Number of fibers after filtering: {len(axon_dicts)}")

--------------------------------------------------------
### Step 2: Define simulation settings
Simulation settings include general simulation parameters such as simulation duration and integration time-step, model-related parameters such as axon model variant and morphological parameter fitting method and stimulation-related parameters such as waveform and amplitude.

#### General parameters

In [ ]:
sim_dur = 5 # milliseconds
sim_dt = 0.005 # milliseconds

#### Model-related Parameters:

In [ ]:
model_variant = "Gaines"
param_fit_method="continuous" # for morphological parameter fitting: "continuous" or "discrete"

The model variant defined above is encoded into two parameters that the simulation pipeline gets, these are `model_type` and `tuned_flag`.
- `model_type` (str or None):
    - "sensory": for forcing sensory variant parameters in gaines and the tuned model.
    - "motor": for forcing motor variant parameters in gaines and the tuned model.
    - "mrg": for forcing MRG original parameters.
    - None: for letting the algorithm decide whether to enforce sensory or motor variant parameters from the axon name for our models and gaines models. If no keywords were found, the default is to use sensory variant parameters. Currently the following keywords are used:
        - Sensory fiber keywords: ["Sensory", "senory", "Aalpha", "Abeta", "\_DR\_", "\_DL\_"]
        - Motor fiber keywords: ["Motor", "motor", "\_alpha", "\_VR\_", "\_VL\_"]
- `tuned_flag` (boolean):
    - Whether to use the tuned model or not, when model_type is not 'MRG'

In [ ]:
# Encode model_variant into model_type and tuned_flag parameters
# This follows the same pattern as in run_titrations.py and run_discrete_simulations.py

if model_variant.lower() == "alashqar":
    model_type = None  # determined from axon name based on keywords
    tuned_flag = True
elif model_variant.lower() == "gaines":
    model_type = None  # determined from axon name based on keywords
    tuned_flag = False
elif model_variant.lower() == "mrg":
    model_type = "MRG"
    tuned_flag = False
else:
    raise ValueError(
        f"Invalid model_variant argument: {model_variant}. Accepted arguments: 'Alashqar', 'Gaines', 'MRG'"
    )

#### Stimulation parameters
here we can define our waveform using parameters such as pulse_width, pulse amplitude, pulse frequency, carrier frequency, ...etc. Helping functions are available under [utils.py](https://gitlab.rrze.fau.de/ProModell/pyns/-/blob/main/utils.py) for creating different kind of waveforms:
- For single-pulse waveforms: use `create_single_pulse()`
- For muti-pulse waveforms having user-defined onsets: use `create_multiple_pulses()`
- For high-frequency waveforms with the possibility of having a carrier frequency: use `create_highfreq_pulse()`

The following example creates a monophasic single-pulse waveform to be used in the simulation

In [ ]:
# it is recommended to use only 1 or -1 for the amplitude since this is intended to only define the polarity of the stimulation.
# Later we will be using another parameter to define the intensity of the stimulation, which would be scaling the defined waveform.
amplitude = 1.0
start_at = 1.0 # milliseconds
pulse_width = 2.0 # milliseconds
end_at = start_at + pulse_width # milliseconds
pulse_shape = "monophasic" # can be "monophasic" or "biphasic"
pulse_biphasic_flag = True if pulse_shape == "biphasic" else False

stim_pulse_t, stim_pulse = create_single_pulse_waveform(
    start_at=start_at,
    end_at=end_at,
    amplitude=amplitude,
    biphasic=pulse_biphasic_flag,
)

# now plot the stimulation pulse
plt.figure(figsize=(10, 4))
plt.plot(stim_pulse_t, stim_pulse, label=f"{pulse_shape} pulse", color='red')
plt.xlabel("Time (ms)")
plt.ylabel("Amplitude (a.u.)")
plt.title("Stimulation Pulse")
plt.show()
plt.close()


#### Step 3: Run a simulation

In [ ]:
# Choose an axon from the filtered dictionary
# Option 1: Select a random axon
axon_info = axon_dicts[np.random.randint(len(axon_dicts))]

# Option 2: Select a specific axon by name (uncomment to use)
# example_axon_name = "S1_DR_..."  # Replace with desired axon name
# axon_info = [axon for axon in axon_dicts if axon['axon_name'] == example_axon_name][0]

print(f"Selected axon: {axon_info['axon_name']}")
print(f"  Diameter: {axon_info['diam']:.2f} μm")
print(f"  Length: {axon_info['length']*1e-3:.2f} mm")

In [ ]:
# Define stimulus intensity
stim_intensity = 50.0 # this will be used to scale the stimulation pulse

In [ ]:
# Initialize the axon model
print("Initializing axon model...")
axon_obj = MyelinatedAxon(
    axon_name=axon_info["axon_name"],
    axon_coords=np.copy(axon_info["points"]),
    fiber_diameter=axon_info["diam"],
    model_type=model_type,
    tuned_model=tuned_flag,
    params_fit_method=param_fit_method,
)

# Initialize NEURON sections
# passive_end_nodes=False uses the default; set to True to have passive end nodes
axon_obj.initialize_neuron(
    passive_end_nodes=True,
)

# Setup recording of membrane voltage
axon_obj.setup_recorders(record_v=True)

# Assign external electric field to the axon
axon_obj.assign_v_ext(field_dict)

# Run the simulation
print("Running simulation...")
axon_res = axon_obj.run_simulation(
    stim_factor=stim_intensity,
    stim_pulse=stim_pulse,
    dt=sim_dt,
    tstop=sim_dur,
    output_path=None,
    return_only_spiking=False,
    exclude_end_node=False,
    prepassive_nodes_as_endnodes=False,
    delete_hoc_objects=False,
)

# Plot membrane potential along the axon
print("Plotting results...")
axon_obj.plot_membrane_potential()
plt.show()

# Clean up NEURON objects to free memory for future simulations
print("Cleaning up...")
axon_obj.delete_recorders()
axon_obj.delete_sections()
print("Done!")

In the plot above, you should see the membrane potential of all nodes of the simulated fiber, with the action potential initiation node highlighted in red. The x-axis shows the position along the axon (normalized 0–1), and the y-axis shows the membrane potential in millivolts.

In the retured `axon_res` you get the general information on the fibers as well as information on spikes and spike initiations. Information on spikes (action potentials) keys:
- 'spike': Information on first spike initiation along the fiber.
- 'spikes_list': Information on all spike initiations if multiple were detected.
- Note: 'spike' and 'spikes_list' might not give an accurate estimate if a node along the fiber spiked more than once over the simulated duration since it relies on NEURON's single spike recorder which records spike times of only the last spike along each node. For more detailed and accurate information on all detected spikes, use 'AP_times' instead.
- 'AP_times': action potential times of all the spikes detected on each of the nodes along the simulation period.

In [ ]:
# Explore the simulation results structure
print("Simulation results keys:")
for key in axon_res.keys():
    print(f"  {key}: {type(axon_res[key])}")

In [ ]:
# Examine spike initiation information
if "spike" in axon_res:
    print("\nFirst spike initiation details:")
    spike_dict = axon_res["spike"]
    for key, value in spike_dict.items():
        print(f"  {key}: {value}")

In [ ]:
# Examine all action potential times detected on each node
print("\nAction potential times on each node:")
for node_idx, spike_times in axon_res["AP_times"].items():
    print(f"  Node {node_idx}: Spike times (ms) = {spike_times}")

##### Visualize the spike initiation along the fiber's trajectory

In [ ]:
fiber_coords = axon_res["segment_midpoints"]
spike_segment_ind = axon_res["spike"]["spike_seg_idx"]
# plot the fiber coordinates
fig, ax = plt.subplots(figsize=(10, 6), subplot_kw={'projection': '3d'})
ax.scatter(fiber_coords[:, 0], fiber_coords[:, 1], fiber_coords[:, 2], c='blue', s=1, label='Fiber Coordinates')
# plot the spike
ax.scatter(fiber_coords[spike_segment_ind, 0], 
           fiber_coords[spike_segment_ind, 1], 
           fiber_coords[spike_segment_ind, 2], 
           c='red', s=50, label='Spikes')
ax.set_xlabel('X (um)')
ax.set_ylabel('Y (um)')
ax.set_zlabel('Z (um)')
x_coords = fiber_coords[:, 0]
y_coords = fiber_coords[:, 1]
z_coords = fiber_coords[:, 2]
ax.set_box_aspect((np.ptp(x_coords), np.ptp(y_coords), np.ptp(z_coords)))  # aspect ratio is 1:1:1 in data space
ax.set_title(f"Axon Coordinates and Spikes for {axon_info['axon_name']}")
ax.legend()
plt.show()
plt.close()
